# Amazon Review - OPTIMIZED PySpark Preprocessing

**Optimizations Applied:**
- Broadcast joins for metadata
- Repartitioning for better parallelism
- Caching strategic DataFrames
- Batch processing for numerical operations
- Optimized text cleaning with native Spark functions
- Predicate pushdown and column pruning

## 1. Setup Spark Session (OPTIMIZED)

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import multiprocessing

# Tự động đếm số lõi CPU thực tế trên máy bạn
cores = multiprocessing.cpu_count()
# Cấu hình số partition thường gấp 2-3 lần số lõi CPU để tối ưu hóa
safe_cores = max(4, int(cores * 0.6))
num_partitions = safe_cores * 3
# OPTIMIZED Spark Configuration
spark = SparkSession.builder \
    .appName("Amazon Review Local Processing") \
    .master(f"local[{safe_cores}]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.sql.autoBroadcastJoinThreshold", "50MB") \
    .config("spark.sql.files.maxPartitionBytes", "128MB") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "200") \
    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
    .config("spark.memory.fraction", "0.6") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "100000") \
    .getOrCreate()
print(f"✅ Spark {spark.version} initialized for LOCAL MODE")
print(f"🖥️  CPU Cores utilized: {cores}")
print(f"📊 Partitions configured: {num_partitions}")
print(f"💾 Driver Memory (Max RAM): 8GB")

✅ Spark 3.5.1 initialized for LOCAL MODE
🖥️  CPU Cores utilized: 16
📊 Partitions configured: 27
💾 Driver Memory (Max RAM): 8GB


## 2. Configuration

In [2]:
from pathlib import Path

ROOT_DIR      = Path().resolve().parent
DATA_DIR      = ROOT_DIR / "data"
RAW_DIR       = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"

# Đảm bảo thư mục processed tồn tại
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 📥 Đường dẫn đầu vào (Ưu tiên dùng Parquet đã convert để nhanh hơn)
REVIEW_RAW_PARQUET = str(RAW_DIR / "Clothing_Shoes_and_Jewelry.parquet")
META_RAW_PARQUET   = str(RAW_DIR / "meta_Clothing_Shoes_and_Jewelry.parquet")

# 📤 Đường dẫn đầu ra cho các bước xử lý tiếp theo
REVIEW_CLEAN_PARQUET = str(PROCESSED_DIR / "review_clean_spark.parquet")
META_CLEAN_PARQUET   = str(PROCESSED_DIR / "meta_clean_spark.parquet")
FINAL_REVIEWS_PARQUET = str(PROCESSED_DIR / "final_kcore_reviews.parquet")
FINAL_META_PARQUET = str(PROCESSED_DIR / "final_kcore_metadata.parquet")

# Các tham số cấu hình khác
K_CORE = 5
OUTLIER_ZSCORE_THRESHOLD = 3.0
MIN_TEXT_LENGTH = 10
MAX_TEXT_LENGTH = 5000
NUM_PARTITIONS = 200
print(f"📂 Project Root: {ROOT_DIR}")
print(f"✅ Đã cấu hình đường dẫn cho dữ liệu Raw và Processed.")

📂 Project Root: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis
✅ Đã cấu hình đường dẫn cho dữ liệu Raw và Processed.


## 3. Optimized Text Cleaning

In [3]:
# ============================================================
# OPTIMIZED TEXT CLEANING - Single Pass for Both Variants
# ============================================================

STOPWORDS_LIST = [
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'he','him','his','she','her','hers','it','its','they','them','their',
    'what','which','who','this','that','these','those','am','is','are','was',
    'were','be','been','being','have','has','had','do','does','did','will',
    'would','could','should','may','might','shall','can','a','an','the',
    'and','but','or','nor','for','so','yet','both','either','neither',
    'not','no','nor','only','own','same','than','too','very','just',
    'because','as','until','while','of','at','by','with','about','against',
    'between','through','during','before','after','above','below','to',
    'from','in','out','on','off','over','under','again','then','once'
]

def apply_optimized_text_cleaning(df, text_col='combined_text'):
    
    print("🚀 Applying optimized text cleaning (single pass for both variants)...")
    
    # Get base text
    base_text = F.coalesce(F.col(text_col), F.lit(""))
    
    # ========================================
    # BERT CLEANING (semantic preserving)
    # ========================================
    bert_clean = base_text
    bert_clean = F.regexp_replace(bert_clean, r'<[^>]+>', ' ')  # HTML tags
    bert_clean = F.regexp_replace(bert_clean, r'http\S+|www\.\S+', ' ')  # URLs
    bert_clean = F.regexp_replace(bert_clean, r'&[a-z]+;', ' ')  # HTML entities
    bert_clean = F.regexp_replace(bert_clean, r'\s+', ' ')  # Whitespace
    bert_clean = F.substring(F.trim(bert_clean), 1, 2500)
    
    # ========================================
    # TF-IDF CLEANING (aggressive)
    # ========================================
    tfidf_clean = F.lower(base_text)
    tfidf_clean = F.regexp_replace(tfidf_clean, r'<[^>]+>', ' ')  # HTML
    tfidf_clean = F.regexp_replace(tfidf_clean, r'[^a-z0-9\s]', ' ')  # Special chars
    
    # Remove stopwords - OPTIMIZED with single regex
    stopwords_pattern = r'\b(' + '|'.join(STOPWORDS_LIST) + r')\b'
    tfidf_clean = F.regexp_replace(tfidf_clean, stopwords_pattern, ' ')
    
    tfidf_clean = F.regexp_replace(tfidf_clean, r'\b\w{1,2}\b', ' ')  # Short words
    tfidf_clean = F.trim(F.regexp_replace(tfidf_clean, r'\s+', ' '))  # Whitespace
    
    # Add both columns in ONE transformation
    result = df.withColumn('text_bert', bert_clean) \
               .withColumn('text_tfidf', tfidf_clean)
    
    print("   ✅ Created 'text_bert' and 'text_tfidf' in SINGLE PASS")
    return result

print("✅ Optimized text cleaning function defined")

✅ Optimized text cleaning function defined


## 4. OPTIMIZED Numerical Processing

In [4]:
# ============================================================
# OPTIMIZED NUMERICAL PROCESSING - Batch Operations
# ============================================================

def optimized_numerical_processing(df, numerical_cols):
    """
    🚀 OPTIMIZED: Batch compute stats and apply transformations
    Instead of multiple passes, compute all stats in ONE aggregation
    """
    print("\n🚀 OPTIMIZED Numerical Processing (batch operations)...")
    
    # ========================================
    # STEP 1: Compute ALL stats in ONE PASS
    # ========================================
    print("\n   Computing statistics in single pass...")
    
    agg_exprs = []
    for col in numerical_cols:
        if col in df.columns:
            agg_exprs.extend([
                F.mean(col).alias(f"{col}_mean"),
                F.stddev(col).alias(f"{col}_stddev"),
                F.expr(f"percentile_approx({col}, 0.5)").alias(f"{col}_median")
            ])
    
    stats = df.select(agg_exprs).collect()[0].asDict()
    
    # ========================================
    # STEP 2: Apply ALL transformations in ONE withColumn chain
    # ========================================
    print("   Applying transformations...")
    
    result = df
    
    for col in numerical_cols:
        if col in df.columns:
            mean = stats[f"{col}_mean"]
            stddev = stats[f"{col}_stddev"]
            median = stats[f"{col}_median"]
            
            # Impute missing with median
            result = result.withColumn(
                col,
                F.coalesce(F.col(col), F.lit(median))
            )
            
            # Add outlier flag (Z-score) in same chain
            if stddev and stddev > 0:
                result = result.withColumn(
                    f"{col}_outlier",
                    F.when(
                        F.abs((F.col(col) - mean) / stddev) > OUTLIER_ZSCORE_THRESHOLD,
                        True
                    ).otherwise(False)
                )
            
            print(f"   ✅ {col}: median={median:.2f}, mean={mean:.2f}, std={stddev:.2f}")
    
    return result

print("✅ Optimized numerical processing function defined")

✅ Optimized numerical processing function defined


## 5. Load & Process Review Data (OPTIMIZED)

In [5]:
# STEP 1: LOAD REVIEW DATA

# Đọc trực tiếp từ Parquet
df_review = spark.read.parquet(REVIEW_RAW_PARQUET)

# 🚀 OPTIMIZATION: Repartition ngay lập tức
df_review = df_review.repartition(NUM_PARTITIONS, "parent_asin", "user_id")

print(f"✅ Loaded {df_review.count():,} reviews from Parquet")

✅ Loaded 66,033,346 reviews from Parquet


## 6. STEP 2-3: Combined Feature Engineering & Text Cleaning (OPTIMIZED)

In [ ]:
# ============================================================
# 🚀 OPTIMIZED STEPS 2 & 3: Combined in ONE transformation chain
# Instead of multiple passes, we do everything in ONE withColumn chain
# ============================================================

print("\n" + "="*60)
print("STEPS 2-3: FEATURE ENGINEERING & TEXT CLEANING (OPTIMIZED)")
print("="*60)

# 🚀 OPTIMIZATION: Chain ALL transformations together
print("\n🚀 Applying ALL transformations in optimized chain...")
df_review = df_review.na.drop(subset=['rating'])
df_review = df_review \
    .withColumn('combined_text',
        F.concat_ws(' ', 
            F.coalesce(F.col('title'), F.lit('')),
            F.coalesce(F.col('text'), F.lit(''))
        )
    ) \
    .withColumn('text_length', F.length(F.col('combined_text'))) \
    .withColumn('review_date', F.from_unixtime(F.col('timestamp') / 1000).cast('timestamp')) \
    .withColumn('year', F.year('review_date')) \
    .withColumn('month', F.month('review_date')) \
    .withColumn('day_of_week', F.dayofweek('review_date')) \
    .withColumn('quarter', F.quarter('review_date')) \
    .withColumn('label',
    F.when(F.col('rating') >= 4.0, 2)      # 2 = Positive
     .when(F.col('rating') <= 2.0, 0)      # 0 = Negative
     .otherwise(1)                         # 1 = Neutral (3.0 sao)
    ).withColumn("label", F.col("label").cast("integer"))\
    .withColumn('is_verified',
        F.coalesce(F.col('verified_purchase'), F.lit(False)).cast('integer')
    ) \
    .withColumn('has_helpful_votes',
        F.when(F.col('helpful_vote') > 0, 1).otherwise(0)
    )

print("✅ Basic features created in optimized chain")

# Filter by text length BEFORE text cleaning (saves processing)
print("\n🚀 Filtering by text length BEFORE cleaning...")
before_filter = df_review.count()
df_review = df_review.filter(
    (F.col('text_length') >= MIN_TEXT_LENGTH) & 
    (F.col('text_length') <= MAX_TEXT_LENGTH)
)
after_filter = df_review.count()
print(f"   Filtered: {before_filter:,} -> {after_filter:,} ({100*(before_filter-after_filter)/before_filter:.2f}% removed)")
print("\n🚀 Ghi dữ liệu trung gian ra file Parquet (Checkpointing)...")

temp_path = str(PROCESSED_DIR / "step2_filtered_temp.parquet")
df_review.write.mode("overwrite").parquet(temp_path)
df_review = spark.read.parquet(temp_path)


STEPS 2-3: FEATURE ENGINEERING & TEXT CLEANING (OPTIMIZED)

🚀 Applying ALL transformations in optimized chain...
✅ Basic features created in optimized chain

🚀 Filtering by text length BEFORE cleaning...
   Filtered: 66,033,346 -> 65,758,104 (0.42% removed)

🚀 Ghi dữ liệu trung gian ra file Parquet (Checkpointing)...


In [7]:
# 🚀 OPTIMIZATION: Cache after filtering, before expensive text cleaning
# df_review = df_review.cache()
# df_review.count()  # Trigger cache
print("   ✅ DataFrame cached before text cleaning")

# Apply optimized text cleaning (single pass for both variants)
print("\n🚀 Applying optimized text cleaning...")
df_review = apply_optimized_text_cleaning(df_review)

print("\n✅ Steps 2-3 complete with OPTIMIZATIONS")

   ✅ DataFrame cached before text cleaning

🚀 Applying optimized text cleaning...
🚀 Applying optimized text cleaning (single pass for both variants)...
   ✅ Created 'text_bert' and 'text_tfidf' in SINGLE PASS

✅ Steps 2-3 complete with OPTIMIZATIONS


## 7. STEP 4: OPTIMIZED Numerical Processing

In [8]:
# ============================================================
# STEP 4: OPTIMIZED NUMERICAL PROCESSING
# ============================================================

print("\n" + "="*60)
print("STEP 4: NUMERICAL PROCESSING (OPTIMIZED)")
print("="*60)

numerical_cols = ['helpful_vote', 'text_length']

# Apply optimized batch processing
df_review = optimized_numerical_processing(df_review, numerical_cols)

print("\n✅ Step 4 complete with batch optimizations")


STEP 4: NUMERICAL PROCESSING (OPTIMIZED)

🚀 OPTIMIZED Numerical Processing (batch operations)...

   Computing statistics in single pass...
   Applying transformations...
   ✅ helpful_vote: median=0.00, mean=0.76, std=9.50
   ✅ text_length: median=112.00, mean=168.92, std=192.76

✅ Step 4 complete with batch optimizations


## 8. STEP 5: Data Quality & Save (OPTIMIZED)

In [ ]:
# ============================================================
# STEP 5: DATA QUALITY CHECKS (OPTIMIZED)
# ============================================================
# bỏ do check nhieu r @@
print("\n" + "="*60)
print("STEP 5: DATA QUALITY CHECKS (OPTIMIZED)")
print("="*60)

critical_cols = ['user_id', 'parent_asin', 'rating', 'text_bert', 'text_tfidf']

# 🚀 OPTIMIZATION: Check all nulls in ONE pass using array operations
print("\n🚀 Checking nulls in single aggregation...")

null_checks = [F.sum(F.col(c).isNull().cast('int')).alias(f"{c}_nulls") 
               for c in critical_cols if c in df_review.columns]

null_stats = df_review.select(null_checks).collect()[0].asDict()

for col, nulls in null_stats.items():
    print(f"   {col.replace('_nulls', '')}: {nulls:,} nulls")

print("\n✅ Step 5 complete")


STEP 5: DATA QUALITY CHECKS (OPTIMIZED)

🚀 Checking nulls in single aggregation...
   user_id: 0 nulls
   parent_asin: 0 nulls
   rating: 0 nulls
   text_bert: 0 nulls
   text_tfidf: 0 nulls

   Removed rows with nulls: 65,758,104 -> 65,758,104 (0 removed)

🚀 Computing distributions...
+------+---------+--------+
|rating|sentiment|   count|
+------+---------+--------+
|   1.0| negative| 5770148|
|   2.0| negative| 3635541|
|   3.0|  neutral| 5515560|
|   4.0| positive| 9014709|
|   5.0| positive|41822146|
+------+---------+--------+


✅ Step 5 complete


In [9]:
# # Remove nulls
# before = df_review.count()
# df_review = df_review.na.drop(subset=critical_cols)
# after = df_review.count()
# print(f"\n   Removed rows with nulls: {before:,} -> {after:,} ({before-after:,} removed)")

# 🚀 OPTIMIZATION: Compute rating and label distribution in ONE aggregation
print("\n🚀 Computing distributions...")
df_review.groupBy('rating', 'label').count() \
    .orderBy('rating').show(15)


🚀 Computing distributions...
+------+-----+--------+
|rating|label|   count|
+------+-----+--------+
|   1.0|    0| 5770148|
|   2.0|    0| 3635541|
|   3.0|    1| 5515560|
|   4.0|    2| 9014709|
|   5.0|    2|41822146|
+------+-----+--------+



In [10]:
# ============================================================
# SAVE PROCESSED REVIEW DATA
# ============================================================

print("\n💾 Saving processed review data...")

# 🚀 OPTIMIZATION: Coalesce partitions before writing to reduce small files
df_review.coalesce(NUM_PARTITIONS // 2) \
    .write.mode('overwrite') \
    .parquet(REVIEW_CLEAN_PARQUET)

# Unpersist cache
df_review.unpersist()
print(f"✅ Review data saved to {REVIEW_CLEAN_PARQUET}")
# print(f"📊 Final shape: {after:,} rows × {len(df_review.columns)} columns")


💾 Saving processed review data...
✅ Review data saved to D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\review_clean_spark.parquet


## 9. Load & Process Metadata (OPTIMIZED)

In [ ]:
# ============================================================
# STEP 6: LOAD & PROCESS METADATA (OPTIMIZED)
# ============================================================

print("\n" + "="*60)
print("STEP 6: METADATA PROCESSING (OPTIMIZED)")
print("="*60)
df_meta = spark.read.parquet(META_RAW_PARQUET)

print(f"✅ Loaded {df_meta.count():,} metadata records")
df_meta = df_meta.filter(F.col("main_category") == "Amazon Fashion")
# 🚀 1. GỌI HÀM CỦA BẠN CHO CỘT PRICE
numerical_cols_meta = ['price', 'average_rating', 'rating_number']
df_meta = optimized_numerical_processing(df_meta, numerical_cols_meta)

print("\n🚀 Processing other metadata columns in optimized chain...")

# 🚀 2. XỬ LÝ CÁC CỘT CÒN LẠI (bỏ 'price' ra khỏi fillna vì hàm trên đã xử lý rồi)
df_meta = df_meta \
    .withColumn('description_text', F.concat_ws(' ', F.col('description'))) \
    .drop('description') \
    .fillna({
        'main_category': 'unknown',
        'store': 'unknown',
        'title': 'unknown',
        'description_text': ''
    }) \
    .withColumn('price_category',
        F.when(F.col('price') < 20, 'budget')
         .when(F.col('price') < 50, 'mid-range')
         .when(F.col('price') < 100, 'premium')
         .otherwise('luxury')
    )
cols_to_drop = [f"{c}_outlier" for c in numerical_cols_meta]
df_meta = df_meta.drop(*cols_to_drop)
print("✅ Metadata processed")

# Save metadata
print(f"\n💾 Saving metadata...")
df_meta.write.mode('overwrite').parquet(META_CLEAN_PARQUET)
print(f"✅ Metadata saved to {META_CLEAN_PARQUET}")


STEP 6: METADATA PROCESSING (OPTIMIZED)
✅ Loaded 7,218,481 metadata records

🚀 OPTIMIZED Numerical Processing (batch operations)...

   Computing statistics in single pass...
   Applying transformations...
   ✅ price: median=22.59, mean=45.02, std=199.59
   ✅ average_rating: median=4.20, mean=4.08, std=0.75
   ✅ rating_number: median=12.00, mean=134.86, std=811.76

🚀 Processing other metadata columns in optimized chain...
✅ Metadata processed

💾 Saving metadata...
✅ Metadata saved to D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\meta_clean_spark.parquet


## 11. OPTIMIZED K-Core Filtering

In [12]:
import pyarrow
import sys
print(f"Python path: {sys.executable}")
print(f"PyArrow version: {pyarrow.__version__}")
print(pyarrow.__version__)

Python path: d:\App\anaconda3\envs\amazon_project\python.exe
PyArrow version: 15.0.2
15.0.2


In [13]:
import pandas as pd
import pyspark.sql.functions as F
import gc
import os

print("\n" + "="*60)
print(f"STEP 8: K-CORE FILTERING (K={K_CORE}) - OPTIMIZED HYBRID")
print("="*60)

df_reviews = spark.read.parquet(REVIEW_CLEAN_PARQUET)
df_meta = spark.read.parquet(META_CLEAN_PARQUET)
df_edges = df_reviews.select('user_id', 'parent_asin').dropDuplicates()
# =========================================================================
# BƯỚC 1: LỌC SƠ BỘ TRÊN SPARK (Giảm data xuống 50-70%)
# =========================================================================
print("🔍 Đang lọc sơ bộ trên Spark để giảm kích thước data...")

# Lọc users/items có ít hơn K_CORE interactions ngay từ đầu
user_counts = df_edges.groupBy('user_id').count()
item_counts = df_edges.groupBy('parent_asin').count()

df_edges_filtered = (
    df_edges
    .join(user_counts.filter(F.col('count') >= K_CORE).select('user_id'), 'user_id')
    .join(item_counts.filter(F.col('count') >= K_CORE).select('parent_asin'), 'parent_asin')
)

# Kiểm tra kích thước sau lọc
n_before = df_edges.count()
n_after = df_edges_filtered.count()
print(f"📉 Đã giảm từ {n_before:,} → {n_after:,} edges ({(1-n_after/n_before)*100:.1f}% reduction)")

# =========================================================================
# BƯỚC 2: KÉO DATA ĐÃ LỌC VỀ PANDAS
# =========================================================================
print("📥 Đang kéo edge list đã lọc về RAM...")
# pdf_edges = df_edges_filtered.toPandas()
# 1. Cho Spark ghi data đã lọc ra một thư mục tạm
temp_dir = str(PROCESSED_DIR /"temp_edges_for_pandas.parquet")
df_edges_filtered.write.mode("overwrite").parquet(temp_dir)

# 2. Dùng trực tiếp Pandas đọc thư mục đó lên (Sử dụng engine pyarrow siêu tốc)
pdf_edges = pd.read_parquet(temp_dir, engine="pyarrow")

# Xóa thư mục tạm cho sạch máy (tùy chọn)
import shutil
shutil.rmtree(temp_dir, ignore_errors=True)
print(f"📋 Số lượng tương tác trong RAM: {len(pdf_edges):,} rows")

# =========================================================================
# BƯỚC 3: CHẠY K-CORE TRÊN PANDAS
# =========================================================================
print("\n🚀 Bắt đầu vòng lặp K-Core trên RAM...")
print("\n🚀 Bắt đầu vòng lặp K-Core trên RAM...")
iteration = 0
while True:
    iteration += 1
    n_before = len(pdf_edges)

    # 1. Tính toán count cho User và Item
    item_counts = pdf_edges['parent_asin'].value_counts()
    user_counts = pdf_edges['user_id'].value_counts()

    # 2. Tìm các ID KHÔNG HỢP LỆ (nhỏ hơn K)
    # Lấy ID bị loại thường ít hơn ID được giữ, giúp .isin() chạy nhẹ hơn
    invalid_items = item_counts[item_counts < K_CORE].index
    invalid_users = user_counts[user_counts < K_CORE].index

    # Nếu không còn ai bị loại -> Hội tụ!
    if len(invalid_items) == 0 and len(invalid_users) == 0:
        print(f"\n✅ Hội tụ sau {iteration} vòng lặp!")
        break

    # 3. Tạo mặt nạ giữ lại những dòng hợp lệ
    # Toán tử ~ nghĩa là "Không nằm trong tập invalid"
    mask = ~(pdf_edges['parent_asin'].isin(invalid_items) | 
             pdf_edges['user_id'].isin(invalid_users))
    
    # Lọc data và copy đè lên để giải phóng bộ nhớ cũ
    pdf_edges = pdf_edges[mask].copy()

    n_after = len(pdf_edges)
    removed = n_before - n_after
    
    print(f"🔄 Vòng {iteration:2d}: {n_before:>12,} → {n_after:>12,} (xóa {removed:>8,})")

# =========================================================================
# BƯỚC 4: ĐƯA KẾT QUẢ TRỞ LẠI SPARK
# =========================================================================
print("\n📤 Đang lưu danh sách ID hợp lệ...")

valid_users_list = pdf_edges['user_id'].unique()
valid_items_list = pdf_edges['parent_asin'].unique()

# Lưu qua Parquet 
pd.DataFrame({'user_id': valid_users_list}).to_parquet("temp_valid_users.parquet", index=False)
pd.DataFrame({'parent_asin': valid_items_list}).to_parquet("temp_valid_items.parquet", index=False)

del pdf_edges, valid_users_list, valid_items_list
gc.collect()

sdf_valid_users = spark.read.parquet("temp_valid_users.parquet")
sdf_valid_items = spark.read.parquet("temp_valid_items.parquet")

# =========================================================================
# BƯỚC 5: KHÔI PHỤC DỮ LIỆU ĐÃ LỌC (NORMALIZED)
# =========================================================================
print("🔄 Đang khôi phục dữ liệu dạng chuẩn hóa (Normalized)...")

# 1. Lọc bảng Reviews
df_final_reviews = (
    df_reviews
    .join(sdf_valid_users, on='user_id', how='inner')
    .join(sdf_valid_items, on='parent_asin', how='inner')
)

# 2. Lọc bảng Metadata (Chỉ lấy info của các sản phẩm qua ải K-Core)
df_final_meta = (
    df_meta
    .join(sdf_valid_items, on='parent_asin', how='inner')
)


STEP 8: K-CORE FILTERING (K=5) - OPTIMIZED HYBRID
🔍 Đang lọc sơ bộ trên Spark để giảm kích thước data...
📉 Đã giảm từ 64,908,952 → 27,675,255 edges (57.4% reduction)
📥 Đang kéo edge list đã lọc về RAM...
📋 Số lượng tương tác trong RAM: 27,675,255 rows

🚀 Bắt đầu vòng lặp K-Core trên RAM...

🚀 Bắt đầu vòng lặp K-Core trên RAM...
🔄 Vòng  1:   27,675,255 →   24,213,540 (xóa 3,461,715)
🔄 Vòng  2:   24,213,540 →   23,240,198 (xóa  973,342)
🔄 Vòng  3:   23,240,198 →   23,028,097 (xóa  212,101)
🔄 Vòng  4:   23,028,097 →   22,959,943 (xóa   68,154)
🔄 Vòng  5:   22,959,943 →   22,944,054 (xóa   15,889)
🔄 Vòng  6:   22,944,054 →   22,938,952 (xóa    5,102)
🔄 Vòng  7:   22,938,952 →   22,937,700 (xóa    1,252)
🔄 Vòng  8:   22,937,700 →   22,937,280 (xóa      420)
🔄 Vòng  9:   22,937,280 →   22,937,184 (xóa       96)
🔄 Vòng 10:   22,937,184 →   22,937,164 (xóa       20)
🔄 Vòng 11:   22,937,164 →   22,937,152 (xóa       12)

✅ Hội tụ sau 12 vòng lặp!

📤 Đang lưu danh sách ID hợp lệ...
🔄 Đang khôi 

## 12. Save Final Dataset

In [15]:
import math
print("\n" + "="*60)
print("STEP 9: SAVE NORMALIZED FINAL DATASETS")
print("="*60)
optimal_partitions_rev = max(1, min(safe_cores, math.ceil(df_final_reviews.count() / 1500000)))

# Lưu Bảng Reviews (Chứa features tương tác & text)
df_final_reviews.coalesce(optimal_partitions_rev) \
    .write.mode('overwrite') \
    .parquet(FINAL_REVIEWS_PARQUET)

# Lưu Bảng Metadata (Chứa thông tin tĩnh của sản phẩm)
# Vì metadata nhỏ hơn nhiều, có thể lưu 1-2 partitions thôi
df_final_meta.coalesce(max(1, safe_cores // 4)) \
    .write.mode('overwrite') \
    .parquet(FINAL_META_PARQUET)

print(f"✅ Đã lưu tập Reviews tại: {FINAL_REVIEWS_PARQUET}")
print(f"✅ Đã lưu tập Metadata tại: {FINAL_META_PARQUET}")


STEP 9: SAVE NORMALIZED FINAL DATASETS
✅ Đã lưu tập Reviews tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\final_kcore_reviews.parquet
✅ Đã lưu tập Metadata tại: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\final_kcore_metadata.parquet


In [ ]:
import shutil
import os

print("\n" + "="*60)
print("STEP 10: CLEAN UP TEMPORARY FILES")
print("="*60)

# Danh sách các file/thư mục tạm cần dọn dẹp
# Bao gồm cả 3 file bạn yêu cầu và 2 file tạm Pandas tạo ra ở bước K-Core
temp_paths_to_delete = [
    str(PROCESSED_DIR / "step2_filtered_temp.parquet"),
    REVIEW_CLEAN_PARQUET,
    META_CLEAN_PARQUET,
    "temp_valid_users.parquet",
    "temp_valid_items.parquet"
]

print("🧹 Đang tiến hành dọn dẹp không gian ổ cứng...\n")

for path in temp_paths_to_delete:
    if os.path.exists(path):
        try:
            # Spark lưu parquet dưới dạng thư mục, Pandas lưu dạng file
            if os.path.isdir(path):
                shutil.rmtree(path)
            else:
                os.remove(path)
            print(f"   ✅ Đã xóa thành công: {path}")
        except Exception as e:
            print(f"   ❌ Lỗi khi xóa {path}: {e}")
    else:
        print(f"   ⏭️ Bỏ qua (không tìm thấy, có thể đã xóa): {path}")

print("\n✅ Hoàn tất quá trình dọn dẹp!")


STEP 10: CLEAN UP TEMPORARY FILES
🧹 Đang tiến hành dọn dẹp không gian ổ cứng...

   ✅ Đã xóa thành công: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\step2_filtered_temp.parquet
   ✅ Đã xóa thành công: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\review_clean_spark.parquet
   ✅ Đã xóa thành công: D:\Data\300_bai_code\300 bai code thieu nhi\amazon-clothing-analysis\data\processed\meta_clean_spark.parquet
   ✅ Đã xóa thành công: temp_valid_users.parquet
   ✅ Đã xóa thành công: temp_valid_items.parquet

✅ Hoàn tất quá trình dọn dẹp!


: 

In [ ]:
# Stop Spark (uncomment to use)
spark.stop()
# print("✅ Spark session stopped")